# Cosmos 3 Nano — MMAD Representative-1400 Optimized (Google Cloud)

Recommended first run: a **2-hour pilot** on a Compute Engine `g2-standard-24`
VM (2× NVIDIA L4, 96 GB RAM) with a persistent disk. A single-L4
`g2-standard-12` is a cheaper fit test, but may use CPU offload.

This notebook runs the immutable 1,400-question representative MMAD subset,
keeps deterministic reasoning, stops after `</think>` plus A/B/C/D, and resumes
from the same GitHub checkpoint namespace as the Kaggle notebook.

Before running, expose `HF_TOKEN` and optionally `GITHUB_TOKEN` as environment
variables or enter them through the hidden prompts. Never paste tokens into a cell.
The working root defaults to `/home/jupyter/jwm-work`; set `JWM_WORK_ROOT` when
using another persistent disk mount.


In [ ]:
import sys, subprocess

packages = [
    'pillow==11.3.0', 'transformers>=5.14.0', 'accelerate',
    'bitsandbytes>=0.49.0', 'qwen-vl-utils', 'safetensors',
    'remotezip', 'requests', 'psutil'
]
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '-U', *packages
], check=True)
print('Dependencies installed. Restart the kernel once, then continue at the environment cell.')


In [ ]:
import os, sys, json, time, shutil, subprocess, random
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

default_root = '/home/jupyter/jwm-work' if Path('/home/jupyter').exists() else '/workspace/jwm-work'
WORK = Path(os.environ.get('JWM_WORK_ROOT', default_root)).expanduser().resolve()
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK / 'hf_cache')

REPO = WORK / 'mini-world-model'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/anhsown/mini-world-model.git', str(REPO)], check=True)

BASE = REPO / 'research/mmad_model_benchmark'
DATA = WORK / 'mmad_full_data'
CACHE = WORK / '.mmad_archive_cache'
OUT = WORK / 'cosmos3_mmad_rep1400_opt256'
SMOKE_OUT = OUT / 'smoke_5.jsonl'
FULL_OUT = OUT / 'predictions_rep1400.jsonl'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
print('persistent work root:', WORK)
print('free disk GiB:', round(shutil.disk_usage(WORK).free / 2**30, 2))
print('repo commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch, transformers, PIL
from PIL import Image, ImageDraw, ImageFont

assert torch.cuda.is_available(), 'Create the VM with an NVIDIA GPU and install the driver.'
gpus = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpus.append({'index': i, 'name': p.name,
                 'vram_GiB': round(p.total_memory / 2**30, 2)})
print(json.dumps({'torch': torch.__version__, 'transformers': transformers.__version__,
                  'pillow': PIL.__version__, 'gpus': gpus}, indent=2))
assert sum(x['vram_GiB'] for x in gpus) >= 24, f'Need at least 24 GiB aggregate VRAM, got {gpus}'
if not any(any(kind in x['name'] for kind in ('L4', 'A100', 'H100', 'H200')) for x in gpus):
    print('WARNING: supported experimentally, but L4/A100/H100/H200 is recommended.')


In [ ]:
# Build canonical metadata, then select the immutable representative subset.
import base64, zlib
from collections import Counter

subprocess.run([
    sys.executable, str(BASE / 'prepare_full.py'),
    '--output', str(DATA), '--metadata-only'
], cwd=BASE, check=True)

from prepare_full import materialize_all_images
full_manifest = json.loads((DATA / 'full_manifest.json').read_text(encoding='utf-8'))
assert len(full_manifest['records']) == 39670

LOCKED_IDS_ZLIB_B64 = 'eNpdnDuyZScMRXOP5iCh32i6XOVy1C/s+VtkXvtmq+DwEUIIAffn5+9/fv375/fvX9/+5q+f//Mxsjk5k1yHPB/4uHKRg+XZFQ6Wb91gd7bPbwlL/mD+K/270t7r/P7eI3zJw/zRbE8epqexvvQAl3E86iqz/ZVsTzXb28H2zMfvR+Q9bP/ZDoCPCbvw/YSbnGSzIUt5YWxPXGHq38mP6SL/ZdYn8j/F8T6VzF8lPNDf09Sv05TvaY7X6WL756D9dgblmXP87HL+7nSS9CYH+2PRaI/lOeRq8rC+cn5fwfzD8XLRH7dA/91vkEOY9sid/fN7mD/CyM3v+2uykSfx/f2Yvgx53kN7dY36ds2CHMKjzO/dhdm/69TPNV9Opn27wfXghqbT/tyS9lYps7z+jjDL7xDOIhd5vo9Mexkf5Rsf7WXIerbdR/1xjOmH4xHmTDfqdxj7E8b5uCzfp3zP+R7O8dnh4ff3Mv1SXyNoDyPikjmfI4+w9De5nkbRXkQZy69meS3j05w/Ifqw05X1iT5ED+ufhH7krrBk9i8/k3SORx75/nA9WXGxvMP1IO18wpfM+ZbG9SHdhKnPOzyQX16un3mpnxm0RynzPYP+zTL7E5wvmVJ+Ur93+WB9LfIayqs+1r+s6UWmvq86fmTKr3ZBBxv9sWVJl/oshIv1i70smW8VXE93uSTnYf0yv6roT5XMp2V+3/RnamhPakrSOZ/643xrmQ8t/vs255JZfjv721fKE3vWl/LtS3vSQX+owySd/vAyy0+uJy3yb/EfO2mPlqU86nM351s39btb5NVS3mYAj8h7aC9axnMZ7Z2P82O3A0WuS+Z4rfqg/Dkcrzm033Noj9e9lfz03+ZwPRrxf8c+ts/o741x/o2sp+OS3zne4yb5OV4j+8sR/2uZ/XPud9YdZrrsHyc4HyboH4/o74T0T/Y/y8yfUl7KeGWGsHzP/coU7eEU948j/t+IvZp2YZFXcz7NfMw/Ml5De7/M8sSfH5k/M9gPne+DfpyP/v/yFcZ+7+x0KzD9/2VXlvxXuNg+7hfObmCEJf0W+xPO9sZl/vyYntKenCvM8gv2ZlnqY3zjMb/vj993mTDzc/+8LP0ZKX9kvAb+7XkKT2b/doCbHMKInyw3vz+wR4vwh5fdyJi/j10Y43FE39Yd4vem+Tmexyn/w/jDC/9Iuimz/55SfoYw5XNduCifC3uzPJ8w2xvSnxB5R0o6x//Q/i4PxydFPinySGf/U8Yzh/pTh+WXyK9oD06LPrazvS3y62b5PUwfqZ/29rHkp/6b2E/7EG9dRvzgGOPDyykMf27V+zPhIlP/zRA/fnyF2X67Ul9IfSnlcXxM5sNWLyz9dY7HMsu7tMd2L+u/Iv8r8rpcDywkfxTzJ+2FJeeDib23ov5bibyrJH+z/qY9M8avl2k/TfTV6O8+Rn+d+//Hku6SzvXdP46Xf7QvLuu9nyNsLP9w/XPRNxd9c7HXq07M79hvLHO8/NL+OeODy/Rv1v2lfG5KOsdj3WGmB8fH82O62MNdDkKY5afUX0eY/pQzHr1M/8HFfu721IUlneuL0z89LvbRxT66+Bc+0h+evyyz/PtxPt2P+rDbaTLPY5ZZ/z1cf+9pyS/1G8f38rxml2vO/yv27V7q56oXvxf/dJONTPt8oySd43cZ71/men3zY/+T/sxNk/RUxnjdon7fEnm2cXw6JZ3294r+XdG3NefKlO+IPGV/s4z2Bs8zltm+OIhHLWN/vEx7Faclne0PWb8lfr7ayvUwZD0LsT8R0h/GK597x/6I/xWZJszyczDeu1w6mfoSYo9C9jtRHL/oj/IXfy6a9jGa9mu7z/bNx/SR/gzt2YrnkOm/pdib5HnKMufXTicnc/4nz4cfSzrtQx7ub5Pnbetu0x6lcf1K4/imc31ebmG2R/y3ZPznJM8fl7n/ySv9vdS/ZLzzMdt3pf8h7Q9pT3B9SfHvllleSvt43vNY0mX8xf4tU35lJSz5uV4lz9+Xqe8p9jVb6pf5lM34Srbol9jj5PnTbs/oz+fQf0mZX8usf6T9Yo+XOR5D/7IYn32M/pbMt+J51SmZD2UpjPjekfONZcYHyrlelEt7eD9hmeNbl/avLvWxLu1tBdfr4vn1MufbMtsb1O8KkW9y/lbJ97Kf2e0M+1fS3qa9KN7/eCz5h+0f6nMN508N9b9EX/qjvrbEL1v0aacP5NkSj+oTwrQnfagfbYhnL9N/aNnftLE/Lfa53Viec/715f6rr/Sf97+W9Xvat77cT3Zwfd/lmPLgedO77sP2ij3tlP4mx79L+lP0f1v8iWY8/oV7WF5z/ux2h98319+W/Xi3jC/vYyyLvGX/vYz2j+jjHOr3HNqf4fntGd6vWaa9HqM/tNsxE2Y6z9OXuX+dS/2dS3swwfm6270g0//d5QLyHFmvhueFj0NY8lN/R+Ll6+6z/SP9GerXcL9q3wf7uuzCsO/LIZySH+1bhv1Zhr7Zx/jJY7bvOL8/OK944Ui255Skl3w/kn+YTvu2HCzPpHzDfuSFQ/m9O9Pvx/7fw/bwvs9yyfct+ZvyD2f7Qvofw/x52J4UfWA8x94FD3DJeHE9fUx5lMivJX9L+7kfsncADh6R54j855P8JvkxX23drSRjftrh+vpY0hGvWKZ+HMYPTe7fLsP+L4fkx/pkR/T/HUiAeb5uLwBPlvoYj1ymvq37wPb6x/7x/Ogxy3OjvNylPKmP8XI792N7Zb6seWD++I4w5RnS3igyz9uXRT8Yb3jM+njebofxgeUULjLPS99xCdvf8n1L+3tcmOMl8+PwfHS5JD/WW3sGn4x49zu+Qf+N8aRltnc/OMJNPiyP+/9l2kNzZ7pzvpiX5Id/+DjIlJ9d6f81Yc4n432nxxg/C2lvUJ+M8dHHlEdS/3f7x/z0N5evMNdzK+l/cf1ecQsb5dtXmeUx/rWcwiKfpj9hjN+bDeen8fzS/KO/IedDy/Qv/GP7XOy5c3++DH/wHReyPJ5fLlM+Lv6DMz5lcp9+mfbReZ/YnPGpZeqT35T8nA8eHA/n+fky7YvzvvFj5k9pf1J/PVuY9manozDXbzkPWm7WJ/6F05/e0aA/c3lf0uT8ZZnyu4fz7zK+ueaB+nJlfb68//aY5TOebpfxzWVpv0v7RT/uNWEPYZZ/aR8u36esuZP+hPSH95OWQ5j+3U3Ohyv6tMu/sEt++jdX1ut1F9i+kva0yLNpH25rfs6Py/Nri4/2IWS/E3xft3yFKR85b1nmeAbfYy1L+eLfLV8y7Wu41Odc7+V85zHkG5f+VHD/vYz4wbKUf+kPL0t5Jekt30t/eN/4LbfsP+97v+sVlK/sp+T9hQXPH03eYyzL+Kb0h+fjJudPy7SHwfj6Mv03OX9a5v5rmf0bae8cSRf9nUt5Df2F/Cjv/I6k07+T9xzLXA+Xm0x5JO93PJbyOR/l/ceytF/mX8r6vcz8xvm/zPYa17t02s+U/VXyvdgy7U/K/igZD12mPiXv3y235Ke+p8wPOZ9apj+f4p8mz2uXuR6k2OfMUpb8tDfJeOu6w0eZ+UvGm+9Fl0V/eB5ryfcXyyKf4fqXMn9yOP+K9/eXaT/rcDyXUV4Z5VvG+b7dvcJDlvIlHlCyfy/ej1/m+CynMNsj+/0Sf7bEX6nL+VC87/SY7blXvqc/WzIf6tJ+V4h8gvO9QsrnedRj+Z77qWX5nvaoJD4n51km74uWuf9admEpT8YzRf9Sxp/x7WUZv+R6K++dlkWevD9uch73uITle9qLEv9tWfIzXlCyvy2Jv1Rz/S6Jv1TL+PI+7m6XvxBm+SPyGZEPz5fXW6N9b1kf+7uSzvWnv5b0lvK5HrXEa5r30ZcZ72yeRy/TfjX/72GZ+4m+lG/LfJb3ZMuczy3+Xwf9hZb4eAfte/O+9+MrLPlF/rLfaJkvXbTPLfEZOR9c5vxuWa9a5kMX/Y9u+b5lPHk/4oVTWP9I//j+ZncHHK+R/fOIfo74byP+24i/NqKvw/8necz0Q/kuSzrn2/A+xbLJ99T3kfi7vGezkfV35DxpJJ45xvj/SHxo3Wu2X/Znw/9TMTkPfcz8fC+3LPXL+j4Snx+ez5u8n3tM+fD9h8n7ueWS8kvKl/649OeKvOV8bS7nx1zq/8j6vyz5Rd94/+Qx84t9Gf6/ybLoUzB+MuI/D+872sh+Us6bl2mPl/m92Jspmc8l+tW0h9OiDy3ylXjv8L3yYxOmvrXMLznfkPeDNsPzAjn/fkz5q/3ifdpl+Jv+/mCJPExn/Pc9R5B01Ofy/0nL8JceS3nD7w3+xPJhfpP6+Z7b5b3h8hzhATMevezM78H2cX4vS3lXvud7sGUpj/HExywvjP3nffLly3TGX5ZFPrwvvFzKzM/7OS7/1+Qf38c+Zn6+13rcwuxvD/OP6BP/n8e/EXnSn/Qj+n3oDz5G/XJ+7Yf/x/C4yS7fp+RPyQ9/3N8DNvIwneunH7738sP3NI+TfISlPzwP9HOlP1wfdnWFf/aY7Y2i/Hi/7DHbw/3gtjZZH+MTfmhv3RgPc+P5xTL13Yz6aS75/Zpwk7F+uvG9kMv7M38OItklnfptPI9epnzk/6iWg8x43TLW392dwB9cb4f64Rf+2OMhU1892D/nfYJl6rOcb72/w2L7muN9eX65THt3+d58m8vyrkt+3n93+f8pv/x/vcdsD88T/fL/DvwynuGX98dd/l9qWdrD/fSKl+Mj/xe1DH9iewf/YJn2P/j/Xe/5mwlrepMRH/YQe7LM/DJ/gvcTHrM+xncfs/3cPy83y7tc7yM4fsH41GOWl1J/Sn28L+byHsfXvWX9RXsXJe2T8Q/en36M8c1D+5PiL8l7kGXa63TKI53rezrtRXJ/48n7fI9Zn8yf5Hv35WH5vP++THuY4q9kUp7J902ePB9fb+0TdsprRB7D7+X/nJa5nhXvY3rx/HuZ7SueX7vEm13eJyxT/nUlf9A+FP8PYzlZP8/bXOKNLvG35ZR07E+9eT/vMepv3mdapn+82y3WJ/agZb2V/2Na5vrYvO+2TH+wg/ZN4l3LtIfN93JbO+d3D+uTeNEy9W0+yn/EH5V4iUu8ZJn2fnietmySTvnt/p/t5fnWSjPiP9nj+ps='
locked_ids = zlib.decompress(base64.b64decode(LOCKED_IDS_ZLIB_B64)).decode().splitlines()
lookup = {row['sample_id']: row for row in full_manifest['records']}
records = [dict(lookup[sid]) for sid in locked_ids]
population = Counter((r['source_dataset'], r['question_type']) for r in full_manifest['records'])
sample_counts = Counter((r['source_dataset'], r['question_type']) for r in records)
for row in records:
    key = (row['source_dataset'], row['question_type'])
    row['sample_weight'] = population[key] / sample_counts[key]

manifest = {
    'benchmark': 'MMAD-Representative',
    'setting': 'locked1400_opt256_reasoning',
    'manifest_sha256': '215b08901829421800d09e626ca8e43e97f2941559b84073da05b844de39d7e4',
    'parent_manifest_sha256': full_manifest['manifest_sha256'],
    'records': records,
}
record_number = {row['sample_id']: i for i, row in enumerate(records, 1)}
assert len(records) == 1400 and len({r['image_file'] for r in records}) == 1400
assert len(sample_counts) == 28 and set(sample_counts.values()) == {50}
assert len({r['category'] for r in records}) == 38
print('parent manifest:', full_manifest['manifest_sha256'])
print('locked subset:', manifest['manifest_sha256'])
print('questions:', len(records), 'unique images:', len({r['image_file'] for r in records}))
print('Validation: 28/28 strata, 38/38 categories, 1,400/1,400 unique images')


In [ ]:
# Read secrets from the VM environment; hidden prompts are only a fallback.
import getpass

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    hf_token = getpass.getpass('HF_TOKEN (hidden, Enter for anonymous): ').strip()
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
print('HF auth:', 'token enabled' if hf_token else 'anonymous')

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    github_token = getpass.getpass('GITHUB_TOKEN (hidden, Enter for read-only): ').strip()
if github_token:
    os.environ['GITHUB_TOKEN'] = github_token
print('GitHub checkpoint push:', 'enabled' if github_token else 'read-only')


In [ ]:
# Load Cosmos 3 Nano Reasoner-only BNB8 across all visible Google Cloud GPUs.
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

MODEL_ID = 'ThePyProgrammer/Cosmos3-Nano-reasoner-bnb8-vllm-und-only'
BASE_MODEL_ID = 'nvidia/Cosmos3-Nano'
MAX_NEW_TOKENS = 256
MAX_RETRY_TOKENS = 128

free_gib = shutil.disk_usage(WORK).free / 2**30
print(f'free disk: {free_gib:.2f} GiB')
assert free_gib >= 8, 'Need at least 8 GiB free for the Reasoner-only checkpoint.'

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28, token=hf_token
)
import psutil

gpu_memory = {}
for i in range(torch.cuda.device_count()):
    total_gib = int(torch.cuda.get_device_properties(i).total_memory / 2**30)
    gpu_memory[i] = f'{max(1, total_gib - 2)}GiB'
cpu_gib = max(16, int(psutil.virtual_memory().total / 2**30) - 12)
max_memory = {**gpu_memory, 'cpu': f'{cpu_gib}GiB'}
print('model max_memory:', max_memory)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    max_memory=max_memory,
    low_cpu_mem_usage=True,
    offload_folder=str(WORK / 'cosmos_offload'),
    offload_state_dict=True,
    attn_implementation='sdpa',
    token=hf_token,
).eval()

print('device map:', json.dumps(getattr(model, 'hf_device_map', {}), indent=2, default=str))
for i in range(2):
    print(f'cuda:{i} allocated GiB:', round(torch.cuda.memory_allocated(i) / 2**30, 2))


In [ ]:
from datetime import datetime, timezone
from common.mmad import (SYSTEM_PROMPT, append_jsonl, load_jsonl, parse_prediction,
                         evaluate_records, write_evaluation)
from PIL import Image
from common.shared_checkpoint import SharedCheckpointStore
from transformers import StoppingCriteria, StoppingCriteriaList
import re

def split_reasoning_response(text):
    cleaned = (text or '').replace('\r\n', '\n').strip()
    if not cleaned:
        return {'reasoning': '', 'response': '', 'parse_format': 'empty'}
    tagged = re.search(r'<think>\s*(.*?)\s*</think>\s*(.*)', cleaned, re.I | re.S)
    if tagged:
        return {'reasoning': tagged.group(1).strip(),
                'response': tagged.group(2).strip(), 'parse_format': 'think_tags'}
    marker = re.search(r'\nResponse\s*\n', cleaned, re.I)
    if marker:
        return {'reasoning': cleaned[:marker.start()].strip(),
                'response': cleaned[marker.end():].strip(), 'parse_format': 'nvidia_ui'}
    prediction = parse_prediction(cleaned)
    return {'reasoning': '', 'response': prediction or cleaned,
            'parse_format': 'answer_only' if prediction else 'unstructured'}

class StopAfterThinkAnswer(StoppingCriteria):
    def __init__(self, tokenizer, prompt_tokens):
        self.tokenizer = tokenizer
        self.prompt_tokens = prompt_tokens

    def __call__(self, input_ids, scores, **kwargs):
        generated = input_ids[0, self.prompt_tokens:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)
        return bool(re.search(r'</think>\s*[A-D](?:\s|$)', text, re.I | re.S))

input_device = model.device

def materialize_batch(selected):
    batch_manifest = {'records': selected}
    materialize_all_images(batch_manifest, DATA, CACHE, range_download=True)
    missing = [r['image_file'] for r in selected if not (DATA / r['image_file']).exists()]
    assert not missing, f'{len(missing)} batch images are missing'

def cleanup_batch(selected):
    for relative in {r['image_file'] for r in selected}:
        (DATA / relative).unlink(missing_ok=True)

def group_by_image_batches(selected, images_per_batch=64):
    groups = {}
    order = []
    for row in selected:
        key = row['image_file']
        if key not in groups:
            groups[key] = []
            order.append(key)
        groups[key].append(row)
    for start in range(0, len(order), images_per_batch):
        keys = order[start:start + images_per_batch]
        yield [row for key in keys for row in groups[key]]

def sync_cuda():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

def ui_style_output(reasoning, response):
    reasoning = (reasoning or '').strip()
    response = (response or '').strip()
    return f'<think>\n{reasoning}\n</think>\n\n{response}'.strip()

def infer_one(sample, max_new_tokens=MAX_NEW_TOKENS, concise=False):
    image_path = (DATA / sample['image_file']).resolve()
    conversation = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': sample['prompt'] + (
    '\nKeep the reasoning under 100 words and answer immediately.' if concise else '')},
        ]},
    ]
    chat = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    with Image.open(image_path) as source:
        image_input = source.convert('RGB')
        inputs = processor(text=[chat], images=[image_input],
                           padding=True, return_tensors='pt').to(input_device)
    seed = 20260731 + int(sample['sample_id'].rsplit('_', 1)[-1])
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    sync_cuda(); started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            stopping_criteria=StoppingCriteriaList([
                StopAfterThinkAnswer(processor.tokenizer, inputs.input_ids.shape[1])
            ]),
        )
    sync_cuda(); latency = time.perf_counter() - started
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)[0].strip()
    parts = split_reasoning_response(raw)
    prediction = parse_prediction(parts['response']) or parse_prediction(raw)
    normalized = ui_style_output(parts['reasoning'], parts['response'])
    return raw, normalized, parts, prediction, latency

def run_range(selected, output_path, label, deadline=None, shared_store=None):
    previous = load_jsonl(output_path)
    done = {r['sample_id'] for r in previous if r.get('status') == 'ok'}
    pending = [r for r in selected if r['sample_id'] not in done]
    print(f'{label}: total={len(selected)} completed={len(selected)-len(pending)} pending={len(pending)}', flush=True)
    all_started = time.perf_counter()
    for position, sample in enumerate(pending, 1):
        if deadline is not None and time.perf_counter() >= deadline:
            print(f'TIME BUDGET REACHED before question {record_number[sample["sample_id"]]}', flush=True)
            break
        try:
            raw, normalized, parts, pred, latency = infer_one(sample)
            attempts = 1
            if not pred:
                raw2, normalized2, parts2, pred2, latency2 = infer_one(
                    sample, max_new_tokens=MAX_RETRY_TOKENS, concise=True
                )
                attempts = 2
                latency += latency2
                if pred2:
                    raw, normalized, parts, pred = raw2, normalized2, parts2, pred2
            status, error = ('ok' if pred else 'parse_failure'), None
        except Exception as exc:
            if isinstance(exc, torch.OutOfMemoryError):
                torch.cuda.empty_cache()
            raw = normalized = ''
            parts = {'reasoning': '', 'response': '', 'parse_format': 'error'}
            pred, latency, attempts, status, error = None, 0.0, 1, 'error', f'{type(exc).__name__}: {exc}'
        row = {
            'sample_id': sample['sample_id'],
            'question_number': record_number[sample['sample_id']],
            'model': MODEL_ID,
            'base_model': BASE_MODEL_ID,
            'precision': 'community BNB8 reasoner-only; optimized deterministic decoding',
            'backend': 'Transformers/device_map=auto/GCP/opt256/deterministic',
            'manifest_sha256': manifest['manifest_sha256'],
            'status': status,
            'prediction': pred,
            'raw_response': raw,
            'ui_style_output': normalized,
            'reasoning': parts['reasoning'],
            'response': parts['response'],
            'parse_format': parts['parse_format'],
            'attempts': attempts,
            'generation_budget': MAX_NEW_TOKENS,
            'latency_seconds': round(latency, 4),
            'error': error,
            'created_at': datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(output_path, row)
        if shared_store is not None:
            shared_store.record(row)
        truth = sample['answer']
        print(f'[{row["question_number"]}/{len(records)}] {status.upper()} pred={pred} truth={truth} '
              f'correct={pred == truth} {latency:.1f}s', flush=True)
        if label == 'SMOKE 1-5':
            print('--- UI-STYLE OUTPUT ---')
            print(normalized or f'ERROR: {error}')
            print('--- PARSED ---')
            print(json.dumps({'reasoning': parts['reasoning'], 'response': parts['response'],
                              'prediction': pred, 'truth': truth}, ensure_ascii=False, indent=2))
        if position % 25 == 0:
            elapsed = time.perf_counter() - all_started
            eta_h = ((len(pending)-position) * elapsed / max(position, 1)) / 3600
            print(f'checkpoint={output_path} ETA={eta_h:.2f}h', flush=True)
    return load_jsonl(output_path)


In [ ]:
# PHASE 1 — run only MMAD questions 1 through 5.
# Inspect all five printed UI-style outputs before running the continuation cell.
smoke_records = records[:5]
materialize_batch(smoke_records)
try:
    smoke_predictions = run_range(smoke_records, SMOKE_OUT, 'SMOKE 1-5')
finally:
    cleanup_batch(smoke_records)

latest = {r['sample_id']: r for r in smoke_predictions}
smoke_latest = [latest[r['sample_id']] for r in smoke_records if r['sample_id'] in latest]
coverage = sum(r.get('status') == 'ok' for r in smoke_latest) / 5
format_rate = sum(r.get('parse_format') == 'think_tags' for r in smoke_latest) / 5
reasoning_rate = sum(bool((r.get('reasoning') or '').strip()) for r in smoke_latest) / 5
SMOKE_GATE = {
    'attempted': len(smoke_latest),
    'parseable_output_coverage': coverage,
    'native_think_tag_rate': format_rate,
    'nonempty_reasoning_rate': reasoning_rate,
    'valid': len(smoke_latest) == 5 and coverage >= 0.8 and reasoning_rate >= 0.8,
    'note': 'ui_style_output is normalized even when the checkpoint emits answer-only text',
}
print(json.dumps(SMOKE_GATE, indent=2))
(OUT / 'smoke_gate.json').write_text(json.dumps(SMOKE_GATE, indent=2), encoding='utf-8')


## Manual smoke gate

Inspect the five outputs. Continue only when image/question pairing is correct, reasoning uses visible evidence, the final response is parseable, and smoke coverage is at least 80%.


In [ ]:
# PHASE 2 — run/resume the locked representative subset.
assert SMOKE_GATE['valid'], f'Smoke gate failed: {SMOKE_GATE}'
MAX_RUNTIME_HOURS = 2.0  # pilot run; change to 24.0 after reviewing speed and quality
started = time.perf_counter()
deadline = started + MAX_RUNTIME_HOURS * 3600

shared_store = SharedCheckpointStore(
    REPO, manifest['manifest_sha256'], 'cosmos3_bnb8_opt256_rep1400_shared',
    push_every=50, token=github_token,
)
print('1. Syncing isolated representative checkpoint...')
shared_store.sync_from_remote()
github_rows = shared_store.successful_rows()
seeded = {row['sample_id']: row for row in github_rows if row.get('status') == 'ok'}

for checkpoint in [FULL_OUT]:
    if not checkpoint.exists():
        continue
    for row in load_jsonl(checkpoint):
        if (row.get('manifest_sha256') == manifest['manifest_sha256']
                and row.get('status') == 'ok'):
            seeded[row['sample_id']] = row

FULL_OUT.parent.mkdir(parents=True, exist_ok=True)
FULL_OUT.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n'
                            for _, row in sorted(seeded.items())), encoding='utf-8')
remaining = [row for row in records if row['sample_id'] not in seeded]
batches = list(group_by_image_batches(remaining, images_per_batch=16))
print(f'resumed={len(seeded)} remaining={len(remaining)} batches={len(batches)}')

for batch_index, batch in enumerate(batches, 1):
    if time.perf_counter() >= deadline:
        print('TIME BUDGET REACHED', flush=True)
        break
    print(f'\n=== BATCH {batch_index}/{len(batches)}: {len(batch)} unique images ===', flush=True)
    materialize_batch(batch)
    try:
        run_range(batch, FULL_OUT, f'Representative batch {batch_index}',
                  deadline=deadline, shared_store=shared_store)
    finally:
        cleanup_batch(batch)

shared_store.flush(push=True)
predictions = load_jsonl(FULL_OUT)
summary, scored = evaluate_records(manifest, predictions)

# Population-weighted accuracy compensates for balanced source×task sampling.
truth = {row['sample_id']: row for row in records}
parsed = [row for row in predictions if row.get('prediction') in {'A','B','C','D'}]
weight_total = sum(truth[row['sample_id']]['sample_weight'] for row in parsed)
summary['weighted_accuracy'] = (
    sum(truth[row['sample_id']]['sample_weight'] *
        (row['prediction'] == truth[row['sample_id']]['answer']) for row in parsed) / weight_total
    if weight_total else None
)
summary['subset_sha256'] = manifest['manifest_sha256']
summary['runtime_budget_hours'] = MAX_RUNTIME_HOURS
summary['actual_runtime_hours'] = round((time.perf_counter() - started) / 3600, 4)
write_evaluation(OUT, summary, scored)
print(json.dumps(summary, ensure_ascii=False, indent=2))
archive = shutil.make_archive(str(WORK / 'cosmos3_mmad_rep1400_opt256_artifacts'), 'zip', OUT)
print('DOWNLOAD:', archive)


In [ ]:
# Rebuild the portable archive at any time.
archive = shutil.make_archive(str(WORK / 'cosmos3_mmad_rep1400_opt256_artifacts'), 'zip', OUT)
print('DOWNLOAD:', archive)
print('size MiB:', round(Path(archive).stat().st_size / 2**20, 2))
print('checkpoint:', FULL_OUT)
